# WCC Data Loading
Loads all data for the Wominjeka Coffee Co vehicle routing project.
- **Real data**: 34 Melbourne cafe locations, distance matrix (km), time matrix (min)
- **Synthetic data**: product catalogue, per-cafe demands, van fleet, perishability params
- **Depot**: Peter Hall Building, UniMelb

Run this block at the top of the main project notebook.

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = 'merged_data'  # adjust path as needed

## Load Products

In [ ]:
products = pd.read_csv(f'{DATA_DIR}/products.csv')
P = products['product_id'].tolist()  # set of products
P_milk = products[products['perishable'] == True]['product_id'].tolist()
P_beans = products[products['perishable'] == False]['product_id'].tolist()

# Parameter dictionaries
revenue = dict(zip(products['product_id'], products['revenue_per_unit']))  # r_p
cost = dict(zip(products['product_id'], products['cost_per_unit']))
margin = dict(zip(products['product_id'], products['margin_per_unit']))
weight = dict(zip(products['product_id'], products['weight_per_unit_kg']))  # w_p

print(f'Products: {len(P)} ({len(P_milk)} milk, {len(P_beans)} beans)')
products

## Load Cafes and Demands

In [ ]:
cafes = pd.read_csv(f'{DATA_DIR}/cafes.csv')
demands_df = pd.read_csv(f'{DATA_DIR}/demands.csv')

# Node mapping: 0 = depot, 1..n = cafes
cafe_ids = cafes['cafe_id'].tolist()
n = len(cafe_ids)
V = list(range(n + 1))  # all nodes
V_cafes = list(range(1, n + 1))  # cafe nodes only

cafe_to_node = {cid: i+1 for i, cid in enumerate(cafe_ids)}
node_to_cafe = {i+1: cid for i, cid in enumerate(cafe_ids)}
node_to_name = {0: 'Depot (Peter Hall)'}
node_to_name.update({i+1: cafes.loc[i, 'cafe_name'] for i in range(n)})

# Demand dictionary: d[i][p] = demand of node i for product p
d = {}
for _, row in demands_df.iterrows():
    node = cafe_to_node[row['cafe_id']]
    if node not in d:
        d[node] = {}
    d[node][row['product_id']] = row['daily_demand']

print(f'Cafes: {n}')
print(f'Nodes: {len(V)} (depot + {n} cafes)')
print(f'\nFirst 5 cafes:')
cafes.head()

## Load Distance and Time Matrices (Real Data)

In [ ]:
dist_df = pd.read_csv(f'{DATA_DIR}/distance_matrix_km.csv', index_col=0)
time_df = pd.read_csv(f'{DATA_DIR}/time_matrix_min.csv', index_col=0)

# Node labels in matrix: 'depot', 'cafe_01', 'cafe_02', ...
node_labels = ['depot'] + cafe_ids

# Travel cost: distance_km * fuel_cost_per_km (use average across fleet)
avg_fuel = 0.45  # $/km average

# c[i,j] = travel cost from node i to node j
c = {}
for i in V:
    for j in V:
        if i != j:
            c[i, j] = round(dist_df.iloc[i, j] * avg_fuel, 4)

# dist_km[i,j] = distance in km
dist_km = {}
for i in V:
    for j in V:
        if i != j:
            dist_km[i, j] = round(dist_df.iloc[i, j], 4)

# t_travel[i,j] = travel time in hours (convert from minutes)
t_travel = {}
for i in V:
    for j in V:
        if i != j:
            t_travel[i, j] = round(time_df.iloc[i, j] / 60, 4)  # min -> hours

# Arcs
A = [(i, j) for i in V for j in V if i != j]

print(f'Arcs: {len(A)}')
print(f'\nSample (depot to Proud Mary Coffee):')
print(f'  Distance: {dist_km[0,1]:.1f} km')
print(f'  Travel time: {t_travel[0,1]*60:.1f} min')
print(f'  Travel cost: ${c[0,1]:.2f}')

## Load Van Fleet

In [ ]:
vans_df = pd.read_csv(f'{DATA_DIR}/vans.csv')

num_vans = 3  # default — change for experiments
K = list(range(num_vans))
Q = {k: vans_df.loc[k, 'capacity_kg'] for k in K}  # Q_k
fuel_cost_per_van = {k: vans_df.loc[k, 'fuel_cost_per_km'] for k in K}

print(f'Using {num_vans} vans')
print(f'Capacities: {Q}')
vans_df.head(num_vans)

## Load Perishability Parameters

In [ ]:
perish_df = pd.read_csv(f'{DATA_DIR}/perishability_params.csv')
perish_params = dict(zip(perish_df['parameter'], perish_df['value']))

T_max = perish_params['T_max_hours']  # hours
service_time_hr = perish_params['service_time_minutes'] / 60  # convert to hours
loading_time_hr = perish_params['loading_time_minutes'] / 60

print(f'T_max: {T_max} hours')
print(f'Service time per cafe: {service_time_hr*60:.0f} min ({service_time_hr:.4f} hr)')
print(f'Loading time at depot: {loading_time_hr*60:.0f} min ({loading_time_hr:.4f} hr)')
perish_df

## Sanity Check

In [ ]:
total_weight = sum(d[i][p] * weight[p] for i in V_cafes for p in P)
total_capacity = sum(Q[k] for k in K)
total_revenue = sum(d[i][p] * revenue[p] for i in V_cafes for p in P)
total_milk_L = sum(d[i][p] for i in V_cafes for p in P_milk)
total_beans_kg = sum(d[i][p] for i in V_cafes for p in P_beans)

print(f'=== Data Summary ===')
print(f'Cafes: {n} | Products: {len(P)} | Vans: {num_vans}')
print(f'\nDemand:')
print(f'  Total milk: {total_milk_L} litres')
print(f'  Total beans: {total_beans_kg} kg')
print(f'  Total weight: {total_weight:.0f} kg')
print(f'  Total revenue: ${total_revenue:.0f}')
print(f'\nCapacity:')
print(f'  Fleet capacity: {total_capacity} kg')
print(f'  Utilisation: {total_weight/total_capacity*100:.0f}%')
print()
if total_weight > total_capacity:
    print('WARNING: Demand exceeds capacity. Not all cafes can be served.')
else:
    print('OK: Capacity sufficient to serve all cafes.')